# Learning Urban Crimes Representation

## Visualización e interpretación del espacio latente

Input:  embeddings_h3_optimal.csv, clusters_h3_optimal.csv, h3_metadata.csv,
        firmas_h3.csv

Output:
  A) Visualización abstracta del espacio latente (UMAP + t-SNE)
     - vis_umap_clusters.html (interactivo)
     - vis_tsne_clusters.html (interactivo)
  B) Visualización geográfica sobre mapa de CDMX
     - mapa_clusters.html (hexágonos coloreados por cluster)
     - mapa_violencia.html (hexágonos coloreados por ratio de violencia)
     - mapa_intensidad.html (hexágonos coloreados por intensidad)

### Paquetes

In [6]:
import pandas as pd
import numpy as np
import h3
from sklearn.manifold import TSNE
import warnings
warnings.filterwarnings('ignore')

import umap

# Gráficos
import folium
from folium.plugins import FloatImage
import branca.colormap as cm

### Ejecución

#### Cargar datos

In [7]:
# Cargar datos
embeddings = pd.read_csv("../data/results/embeddings_h3_optimal.csv", index_col='h3_id')
clusters = pd.read_csv("../data/results/clusters_h3_optimal.csv", index_col='h3_id')
metadata = pd.read_csv("../data/auxiliar/h3_metadata.csv", index_col='h3_id')
firmas = pd.read_csv("../data/auxiliar/firmas_h3.csv", index_col='h3_id')

# Unir todo
df = embeddings.join(clusters).join(metadata).join(firmas[['intensidad_log', 'ratio_violencia']])

print(f"Hexágonos: {len(df)}")
print(f"Columnas: {df.shape[1]}")

Hexágonos: 1061
Columnas: 22


#### Implementación

In [ ]:
# ============================================================================
# PASO 1: UMAP
# ============================================================================

emb_cols = [c for c in embeddings.columns if c.startswith('emb_')]
X_emb = df[emb_cols].values

# UMAP con parámetros robustos
reducer = umap.UMAP(
    n_components=2,
    n_neighbors=15,
    min_dist=0.1,
    metric='euclidean',
    random_state=42
)
umap_2d = reducer.fit_transform(X_emb)
df['umap_x'] = umap_2d[:, 0]
df['umap_y'] = umap_2d[:, 1]
print(f"UMAP completado: {umap_2d.shape}")

# ============================================================================
# PASO 2: t-SNE
# ============================================================================

tsne = TSNE(
    n_components=2,
    perplexity=30,
    max_iter=1000,
    random_state=42,
    init='pca'
)
tsne_2d = tsne.fit_transform(X_emb)
df['tsne_x'] = tsne_2d[:, 0]
df['tsne_y'] = tsne_2d[:, 1]
print(f"t-SNE completado: {tsne_2d.shape}")

UMAP completado: (1061, 2)
  t-SNE completado: (1061, 2)


In [9]:
# ============================================================================
# PASO 3: Visualizaciones abstractas con Plotly (interactivas)
# ============================================================================

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Preparar datos para hover
df['cluster_str'] = df['cluster_kmeans'].astype(str)
df['intensidad_real'] = np.expm1(df['intensidad_log']).astype(int)

# Delito dominante por hexágono
delito_cols = [c for c in firmas.columns if c.startswith('delito_')]
df['delito_dominante'] = firmas[delito_cols].idxmax(axis=1).str.replace('delito_', '').str.replace('_', ' ').str.upper()

hover_template = (
    "<b>%{customdata[0]}</b><br>"
    "Colonia: %{customdata[1]}<br>"
    "Cluster: %{customdata[2]}<br>"
    "Registros: %{customdata[3]:,}<br>"
    "Ratio violencia: %{customdata[4]:.3f}<br>"
    "Delito dominante: %{customdata[5]}<br>"
    "<extra></extra>"
)
customdata = df[['alcaldia_dominante', 'colonia_dominante', 'cluster_str',
                  'intensidad_real', 'ratio_violencia', 'delito_dominante']].values

# --- 3a. UMAP coloreado por cluster ---
fig_umap_cluster = px.scatter(
    df, x='umap_x', y='umap_y',
    color='cluster_str',
    hover_data=['alcaldia_dominante', 'colonia_dominante', 'intensidad_real', 'ratio_violencia'],
    title='Espacio Latente UMAP — Clusters K-Means (GAE óptimo)',
    labels={'umap_x': 'UMAP 1', 'umap_y': 'UMAP 2', 'cluster_str': 'Cluster'},
    width=900, height=700,
    color_discrete_sequence=px.colors.qualitative.Set3
)
fig_umap_cluster.update_traces(marker=dict(size=6, opacity=0.8))
fig_umap_cluster.update_layout(legend_title_text='Cluster')
fig_umap_cluster.write_html('../visualizations/vis_umap_clusters.html')
print(f"/visualizations/vis_umap_clusters.html")

# --- 3b. UMAP coloreado por alcaldía ---
fig_umap_alc = px.scatter(
    df, x='umap_x', y='umap_y',
    color='alcaldia_dominante',
    hover_data=['colonia_dominante', 'cluster_str', 'intensidad_real', 'ratio_violencia'],
    title='Espacio Latente UMAP — Por Alcaldía',
    labels={'umap_x': 'UMAP 1', 'umap_y': 'UMAP 2', 'alcaldia_dominante': 'Alcaldía'},
    width=900, height=700,
    color_discrete_sequence=px.colors.qualitative.Dark24
)
fig_umap_alc.update_traces(marker=dict(size=6, opacity=0.8))
fig_umap_alc.write_html('../visualizations/vis_umap_alcaldias.html')
print(f"/visualizations/vis_umap_alcaldias.html")

# --- 3c. UMAP coloreado por ratio de violencia ---
fig_umap_viol = px.scatter(
    df, x='umap_x', y='umap_y',
    color='ratio_violencia',
    hover_data=['alcaldia_dominante', 'colonia_dominante', 'cluster_str', 'intensidad_real'],
    title='Espacio Latente UMAP — Ratio de Violencia',
    labels={'umap_x': 'UMAP 1', 'umap_y': 'UMAP 2', 'ratio_violencia': 'Ratio Violencia'},
    width=900, height=700,
    color_continuous_scale='RdYlBu_r'  # Rojo = más violencia
)
fig_umap_viol.update_traces(marker=dict(size=6, opacity=0.8))
fig_umap_viol.write_html('../visualizations/vis_umap_violencia.html')
print(f"/visualizations/vis_umap_violencia.html")

# --- 3d. UMAP coloreado por intensidad ---
fig_umap_int = px.scatter(
    df, x='umap_x', y='umap_y',
    color='intensidad_log',
    hover_data=['alcaldia_dominante', 'colonia_dominante', 'cluster_str', 'intensidad_real'],
    title='Espacio Latente UMAP — Intensidad (log registros)',
    labels={'umap_x': 'UMAP 1', 'umap_y': 'UMAP 2', 'intensidad_log': 'Intensidad (log)'},
    width=900, height=700,
    color_continuous_scale='Viridis'
)
fig_umap_int.update_traces(marker=dict(size=6, opacity=0.8))
fig_umap_int.write_html('../visualizations/vis_umap_intensidad.html')
print(f"/visualizations/vis_umap_intensidad.html")

# --- 3e. t-SNE coloreado por cluster ---
fig_tsne = px.scatter(
    df, x='tsne_x', y='tsne_y',
    color='cluster_str',
    hover_data=['alcaldia_dominante', 'colonia_dominante', 'intensidad_real', 'ratio_violencia'],
    title='Espacio Latente t-SNE — Clusters K-Means (GAE óptimo)',
    labels={'tsne_x': 't-SNE 1', 'tsne_y': 't-SNE 2', 'cluster_str': 'Cluster'},
    width=900, height=700,
    color_discrete_sequence=px.colors.qualitative.Set3
)
fig_tsne.update_traces(marker=dict(size=6, opacity=0.8))
fig_tsne.write_html('../visualizations/vis_tsne_clusters.html')
print(f"/visualizations/vis_tsne_clusters.html")

/visualizations/vis_umap_clusters.html
/visualizations/vis_umap_alcaldias.html
/visualizations/vis_umap_violencia.html
/visualizations/vis_umap_intensidad.html
/visualizations/vis_tsne_clusters.html


In [12]:
# ============================================================================
# PASO 4: Visualizaciones geográficas con Folium
# ============================================================================

# Centro de CDMX
CDMX_CENTER = [19.38, -99.14]

# Función para obtener bordes del hexágono H3
def h3_to_polygon(h3_id):
    """Retorna lista de [lat, lng] para el borde del hexágono."""
    boundary = h3.cell_to_boundary(h3_id)
    # h3 retorna (lat, lng), folium necesita [lat, lng]
    return [[lat, lng] for lat, lng in boundary]

# --- 4a. Mapa de clusters ---

# Paleta de colores para 15 clusters
cluster_colors = [
    '#e6194b', '#3cb44b', '#ffe119', '#4363d8', '#f58231',
    '#911eb4', '#46f0f0', '#f032e6', '#bcf60c', '#fabebe',
    '#008080', '#e6beff', '#9a6324', '#800000', '#aaffc3'
]

m_clusters = folium.Map(location=CDMX_CENTER, zoom_start=11, tiles='CartoDB positron')

for h3_id, row in df.iterrows():
    polygon = h3_to_polygon(h3_id)
    cluster = int(row['cluster_kmeans'])
    color = cluster_colors[cluster % len(cluster_colors)]

    popup_html = f"""
    <b>Cluster {cluster}</b><br>
    Alcaldía: {row['alcaldia_dominante']}<br>
    Colonia: {row['colonia_dominante']}<br>
    Registros: {int(row['intensidad_real']):,}<br>
    Ratio violencia: {row['ratio_violencia']:.3f}<br>
    Delito dominante: {row['delito_dominante']}
    """

    folium.Polygon(
        locations=polygon,
        color=color,
        weight=1,
        fill=True,
        fill_color=color,
        fill_opacity=0.6,
        popup=folium.Popup(popup_html, max_width=300),
        tooltip=f"Cluster {cluster} | {row['alcaldia_dominante']}"
    ).add_to(m_clusters)

# Leyenda
legend_html = '<div style="position:fixed;bottom:50px;left:50px;z-index:1000;background:white;padding:10px;border-radius:5px;border:1px solid grey;">'
legend_html += '<b>Clusters K-Means</b><br>'
for i in range(min(15, len(df['cluster_kmeans'].unique()))):
    legend_html += f'<i style="background:{cluster_colors[i]};width:12px;height:12px;display:inline-block;margin-right:5px;"></i> Cluster {i}<br>'
legend_html += '</div>'
m_clusters.get_root().html.add_child(folium.Element(legend_html))

m_clusters.save('../visualizations/mapa_clusters.html')
print(f"/visualizations/mapa_clusters.html")

# --- 4b. Mapa de ratio de violencia ---

m_violencia = folium.Map(location=CDMX_CENTER, zoom_start=11, tiles='CartoDB dark_matter')

# Colormap: azul (baja violencia) → rojo (alta violencia)
vmin, vmax = df['ratio_violencia'].quantile(0.05), df['ratio_violencia'].quantile(0.95)
colormap_viol = cm.LinearColormap(
    colors=['#2166ac', '#67a9cf', '#fddbc7', '#ef8a62', '#b2182b'],
    vmin=vmin, vmax=vmax,
    caption='Ratio de Violencia'
)

for h3_id, row in df.iterrows():
    polygon = h3_to_polygon(h3_id)
    ratio = row['ratio_violencia']
    color = colormap_viol(min(max(ratio, vmin), vmax))

    popup_html = f"""
    <b>Ratio violencia: {ratio:.3f}</b><br>
    Alcaldía: {row['alcaldia_dominante']}<br>
    Colonia: {row['colonia_dominante']}<br>
    Registros: {int(row['intensidad_real']):,}<br>
    Cluster: {int(row['cluster_kmeans'])}
    """

    folium.Polygon(
        locations=polygon,
        color=color,
        weight=1,
        fill=True,
        fill_color=color,
        fill_opacity=0.7,
        popup=folium.Popup(popup_html, max_width=300),
        tooltip=f"Violencia: {ratio:.3f} | {row['alcaldia_dominante']}"
    ).add_to(m_violencia)

colormap_viol.add_to(m_violencia)
m_violencia.save('../visualizations/mapa_violencia.html')
print(f"/visualizations/mapa_violencia.html")

# --- 4c. Mapa de intensidad ---

m_intensidad = folium.Map(location=CDMX_CENTER, zoom_start=11, tiles='CartoDB positron')

vmin_int = df['intensidad_log'].quantile(0.05)
vmax_int = df['intensidad_log'].quantile(0.95)
colormap_int = cm.LinearColormap(
    colors=['#ffffcc', '#a1dab4', '#41b6c4', '#2c7fb8', '#253494'],
    vmin=vmin_int, vmax=vmax_int,
    caption='Intensidad (log registros)'
)

for h3_id, row in df.iterrows():
    polygon = h3_to_polygon(h3_id)
    intens = row['intensidad_log']
    color = colormap_int(min(max(intens, vmin_int), vmax_int))

    popup_html = f"""
    <b>Registros: {int(row['intensidad_real']):,}</b><br>
    Alcaldía: {row['alcaldia_dominante']}<br>
    Colonia: {row['colonia_dominante']}<br>
    Ratio violencia: {row['ratio_violencia']:.3f}<br>
    Cluster: {int(row['cluster_kmeans'])}
    """

    folium.Polygon(
        locations=polygon,
        color=color,
        weight=1,
        fill=True,
        fill_color=color,
        fill_opacity=0.7,
        popup=folium.Popup(popup_html, max_width=300),
        tooltip=f"{int(row['intensidad_real']):,} registros | {row['alcaldia_dominante']}"
    ).add_to(m_intensidad)

colormap_int.add_to(m_intensidad)
m_intensidad.save('../visualizations/mapa_intensidad.html')
print(f"/visualizations/mapa_intensidad.html")

# --- 4d. Mapa de delito dominante ---

m_delito = folium.Map(location=CDMX_CENTER, zoom_start=11, tiles='CartoDB positron')

# Colores por delito dominante
delitos_unicos = df['delito_dominante'].unique()
delito_color_map = {}
palette = px.colors.qualitative.Set3 + px.colors.qualitative.Pastel1
for i, d in enumerate(delitos_unicos):
    delito_color_map[d] = palette[i % len(palette)]

for h3_id, row in df.iterrows():
    polygon = h3_to_polygon(h3_id)
    delito = row['delito_dominante']
    color = delito_color_map[delito]

    popup_html = f"""
    <b>{delito}</b><br>
    Alcaldía: {row['alcaldia_dominante']}<br>
    Colonia: {row['colonia_dominante']}<br>
    Registros: {int(row['intensidad_real']):,}<br>
    Ratio violencia: {row['ratio_violencia']:.3f}
    """

    folium.Polygon(
        locations=polygon,
        color=color,
        weight=1,
        fill=True,
        fill_color=color,
        fill_opacity=0.6,
        popup=folium.Popup(popup_html, max_width=300),
        tooltip=f"{delito} | {row['alcaldia_dominante']}"
    ).add_to(m_delito)

m_delito.save('../visualizations/mapa_delito_dominante.html')
print(f"/visualizations/mapa_delito_dominante.html")

/visualizations/mapa_clusters.html
/visualizations/mapa_violencia.html
/visualizations/mapa_intensidad.html
/visualizations/mapa_delito_dominante.html


In [14]:
# ============================================================================
# PASO 5: Exportar coordenadas de proyección
# ============================================================================

proyecciones = df[['umap_x', 'umap_y', 'tsne_x', 'tsne_y']].copy()
proyecciones.index.name = 'h3_id'
proyecciones.to_csv('../data/vis_results/proyecciones_2d.csv', encoding='utf-8-sig')
print(f"/data/vis_results/proyecciones_2d.csv")

/data/vis_results/proyecciones_2d.csv
